# Bài 12 — Report Agent & Hoàn thiện

## Lý thuyết

### Vấn đề cần giải quyết

Ở Bài 11, Orchestrator đã điều phối được 4 agent, và `analysis_node` tạo ra `final_result` — nhưng đó là **1 đoạn text tự do** (LLM viết sao thì ra vậy), chưa có cấu trúc cố định. Với một hệ thống báo cáo thật, người dùng cần 1 format **nhất quán, dễ đọc**, dù câu hỏi là gì.

**Report Agent** là node cuối cùng: nhận toàn bộ dữ liệu đã thu thập + kết quả phân tích, rồi xuất ra theo đúng 4 phần cố định: **Tóm tắt → Dữ liệu → Phân tích → Khuyến nghị**.

### Vấn đề thứ 2: Error handling cho toàn hệ thống

Ở Bài 11, bạn đã gặp thật các lỗi 503/429 từ Gemini API. Hiện tại, nếu 1 agent bất kỳ (ví dụ `rag_agent`) bị lỗi giữa luồng, exception đó sẽ **lan ra ngoài và làm sập toàn bộ graph** — người dùng không nhận được gì cả, dù 2 agent khác đã chạy thành công và có dữ liệu tốt.

**Nguyên tắc graceful degradation** (suy giảm có kiểm soát): nếu 1 phần hệ thống lỗi, các phần còn lại vẫn nên hoạt động bình thường, và hệ thống nên **báo rõ phần nào bị thiếu** thay vì crash hoàn toàn.

| Khái niệm | Ý nghĩa |
|---|---|
| `try / except` | Cú pháp Python: chạy code trong `try`, nếu có lỗi (exception) thì nhảy vào `except` xử lý, **không làm crash chương trình** |
| Áp dụng vào node | Mỗi node wrapper (`web_research_node`, `financial_data_node`, `rag_node`, `analysis_node`) bọc lệnh gọi agent trong `try/except` — lỗi thì trả về 1 thông báo lỗi rõ ràng vào field state, thay vì để exception ném ra ngoài làm graph dừng đột ngột |

> So sánh: không có `try/except` giống 1 dây chuyền sản xuất mà 1 trạm hỏng thì dừng cả nhà máy. Có `try/except`, trạm hỏng chỉ báo "trạm này lỗi", còn các trạm khác vẫn chạy tiếp.

### Kiến trúc hệ thống hoàn chỉnh (sau Bài 12)

```
                    ┌──────────────────┐
                    │   router_node    │
                    └─────────┬────────┘
              ┌───────────────┼───────────────┐
              ▼               ▼               ▼
      ┌───────────────┐┌────────────┐┌───────────┐
      │ Web Research  ││ Financial  ││    RAG    │   ← mỗi node có
      │     node      ││ Data node  ││   node    │     try/except riêng
      └───────┬───────┘└─────┬──────┘└─────┬─────┘
              └──────────────┴─────────────┘
                             ▼
                    ┌──────────────────┐
                    │  analysis_node    │   ← cũng có try/except
                    └─────────┬────────┘
                              ▼
                    ┌──────────────────┐
                    │   report_node     │   ← MỚI: format báo cáo
                    │                   │     4 phần cố định
                    └─────────┬────────┘
                              ▼
                             END
```

`report_node` là node **duy nhất mới** so với Bài 11. Phần error handling áp dụng cho **toàn bộ các node cũ** (không thêm node mới nào cho việc này).

## Ví dụ — try/except trong 1 node

Mô phỏng 1 node có thể lỗi (chia cho 0), để thấy graph **không sập** nhờ `try/except` — node lỗi chỉ ghi lại thông báo lỗi vào state, graph vẫn chạy tiếp tới node sau.

In [1]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict


class DivideState(TypedDict):
    a: float
    b: float
    result: str


def risky_divide_node(state: DivideState):
    try:
        value = state["a"] / state["b"]
        return {"result": f"Kết quả: {value}"}
    except ZeroDivisionError as e:
        print(f"[Risky Node] Lỗi: {e}")
        return {"result": "Lỗi: không thể chia cho 0"}


def print_result_node(state: DivideState):
    print(f"[Print] {state['result']}")
    return {}

In [2]:
graph_builder = StateGraph(DivideState)
graph_builder.add_node("risky_divide", risky_divide_node)
graph_builder.add_node("print_result", print_result_node)

graph_builder.add_edge(START, "risky_divide")
graph_builder.add_edge("risky_divide", "print_result")
graph_builder.add_edge("print_result", END)

graph = graph_builder.compile()

In [3]:
final_state = graph.invoke({"a": 10.0, "b": 2.0, "result": ""})
print("State cuối cùng:", final_state)

# b = 0 -> gây lỗi, nhưng graph KHÔNG sập nhờ try/except
final_state2 = graph.invoke({"a": 10.0, "b": 0.0, "result": ""})
print("State cuối cùng:", final_state2)

[Print] Kết quả: 5.0
State cuối cùng: {'a': 10.0, 'b': 2.0, 'result': 'Kết quả: 5.0'}
[Risky Node] Lỗi: float division by zero
[Print] Lỗi: không thể chia cho 0
State cuối cùng: {'a': 10.0, 'b': 0.0, 'result': 'Lỗi: không thể chia cho 0'}


## Bài tập

Hoàn thiện hệ thống: thêm `report_node` + error handling cho toàn bộ graph ở Bài 11.

1. **Copy toàn bộ code** từ `lesson11_orchestrator.ipynb` (phần bài tập đã hoàn chỉnh) vào notebook này: 4 hàm agent, `OrchestratorState`, `router_node`, `route_to_agents`, 3 node wrapper, `analysis_node`, và phần build graph.

2. **Thêm field mới** vào `OrchestratorState`: `report: str` (kết quả cuối cùng, đã format).

3. **Thêm `try/except` vào 4 node wrapper đã có** (`web_research_node`, `financial_data_node`, `rag_node`, `analysis_node`): bọc lệnh gọi agent (`web_research_agent(...)`, v.v.) trong `try`, nếu lỗi (`except Exception as e`) thì trả về field tương ứng với nội dung báo lỗi rõ ràng, ví dụ `{"web_result": f"[Lỗi] Không lấy được tin tức: {e}"}` — **không để exception ném ra ngoài**.

4. Viết **`report_agent(question, data)`**: nhận câu hỏi gốc và `data` (kết quả từ `analysis_node`), yêu cầu LLM xuất báo cáo theo đúng 4 phần cố định:
   - **Tóm tắt**: 1-2 câu tổng quan
   - **Dữ liệu**: các số liệu chính đã thu thập được
   - **Phân tích**: nhận định/xu hướng
   - **Khuyến nghị**: gợi ý hành động (mua/bán/theo dõi thêm...)
   Nhắc lại guard chống hallucination giống các agent khác (chỉ dùng dữ liệu được cung cấp).

5. Viết **`report_node(state)`**: bọc `report_agent(...)` trong `try/except` (giống bước 3), gọi với `state["question"]` và `state["final_result"]`, trả về `{"report": ...}`.

6. **Sửa lại graph đã build ở Bài 11**:
   - `add_node("report", report_node)`
   - Đổi `add_edge("analysis", END)` thành `add_edge("analysis", "report")`
   - Thêm `add_edge("report", END)`

7. **Test toàn hệ thống** với nhiều câu hỏi khác nhau (yêu cầu của Bài 12 là test kỹ, không chỉ 1-2 câu):
   - Vài câu hỏi bình thường (như Bài 11) để xem `report` có đúng format 4 phần không.
   - **Test error handling thật**: tạm sửa 1 chỗ để gây lỗi có chủ đích (ví dụ gọi `financial_data_agent(None)` thay vì symbol thật trong 1 lần test riêng, hoặc tạm đổi tên model thành 1 chuỗi sai để mô phỏng lỗi gọi LLM) — chạy `graph.invoke(...)` và xác nhận **graph không sập**, `report` vẫn xuất ra (có ghi rõ phần nào bị lỗi), rồi sửa lại đúng như cũ sau khi test xong.

Viết code của bạn vào các ô dưới đây:

In [15]:
import os
from dotenv import load_dotenv
from google import genai
from ddgs import DDGS


load_dotenv()
client = genai.Client(api_key=os.getenv("GOOGLE_API_KEY"))

In [16]:
# 1. Copy toàn bộ code từ Bài 11: import, 4 hàm agent

# 1. web_research_agent
def web_research_agent(query):
    # Bước 1: Tìm kiếm tin tức thật
    with DDGS() as ddgs:
        results = ddgs.text(query, max_results=3)

    # Bước 2: Ghép các kết quả tìm được thành context
    news_context = "\n\n".join([f"{r['title']}: {r['body']}" for r in results])

    # Bước 3: Đưa cho LLM tổng hợp thành câu trả lời tự nhiên
    prompt = f"""Dựa vào các tin tức sau:

{news_context}

Hãy tóm tắt thông tin quan trọng nhất."""

    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=prompt
    )
    return response.text
import yfinance as yf

# 2. financial_data_agent
def financial_data_agent(symbol):
    ticker_obj = yf.Ticker(symbol)
    info = ticker_obj.info
    return {
        "symbol": symbol,
        "currentPrice": info.get("currentPrice", "Không tìm thấy giá cổ phiếu"),
        "marketCap": info.get("marketCap", "Không tìm thấy marketCap"),
        "trailingPE": info.get("trailingPE", "Không tìm thấy trailingPE")
    }

# 3. rag_agent
from sentence_transformers import SentenceTransformer
import chromadb

embed_model = SentenceTransformer("all-MiniLM-L6-v2")
db_client = chromadb.PersistentClient(path=r"D:\Code\AI\Multi-Agent Research Assistant project\phase2\chroma_db")
collection = db_client.get_or_create_collection(name="nvidia_report")

def rag_agent(question):
    question_embedding = embed_model.encode(question)
    results = collection.query(
        query_embeddings=[question_embedding.tolist()],
        n_results=3
    )
    chunks = results["documents"][0]
    context = "\n\n".join(chunks)

    prompt = f"""Dựa vào thông tin sau từ báo cáo tài chính NVIDIA:

{context}

Hãy trả lời câu hỏi: {question}
Nếu thông tin không đủ để trả lời, hãy nói rõ là không tìm thấy thông tin liên quan.
    """

    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=prompt
    )
    return response.text


# 4. analysis_agent
def analysis_agent(data):
    prompt = f"""Dựa vào thông tin tổng hợp sau về tài chính NVIDIA:

{data}

Hãy tổng hợp thông tin, phân tích xu hướng. 
Chỉ sử dụng dữ liệu được cung cấp ở trên. Không tự thêm số liệu, sự kiện hay bất kì thông tin nào không có trong dữ liệu này.
    """
    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=prompt
    )
    return response.text

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7872.68it/s]


In [17]:
# 2. OrchestratorState (thêm field "report")

from langgraph.graph import StateGraph, START, END
from typing import TypedDict, List

class OrchestratorState(TypedDict):
    question: str
    symbol: str
    needed_agents: List[str]  # Ví dụ ["financial_data", "rag"]
    web_result: str
    financial_result: dict
    rag_result: str
    final_result: str
    report: str

In [18]:
# router_node + route_to_agents (copy nguyên từ Bài 11)

def router_node(state: OrchestratorState):
    question = state['question'].lower()
    
    agents = []

    if "tin tức" in question or "news" in question:
        agents.append("web_research")
    
    if "giá" in question or "price" in question:
        agents.append("financial_data")
    
    if "báo cáo" in question or "doanh thu" in question or "report" in question or "revenue" in question:
        agents.append("rag")

    if not agents:
        agents = ["web_research", "financial_data", "rag"]

    print(f"[Router] Agent cần chạy: {agents}")

    return {"needed_agents": agents}


def route_to_agents(state: OrchestratorState):
    return state["needed_agents"]

In [19]:
# 3. Thêm try/except vào 3 node wrapper: web_research_node, financial_data_node, rag_node
def web_research_node(state: OrchestratorState):
    print("[Web Research Node] đang chạy...")
    try:
        result = web_research_agent(state['question'])
        return {"web_result": result}
    except Exception as e:
        return {"web_result": f"[Lỗi] Không lấy được tin tức: {e}"}

def financial_data_node(state: OrchestratorState):
    print("[Financial Data Node] đang chạy...")
    try:
        result = financial_data_agent(state['symbol'])
        return {"financial_result": result}
    except Exception as e:
        return {"financial_result": f"[Lỗi] Không lấy có dữ liệu: {e}"}

def rag_node(state: OrchestratorState):
    print("[Rag Node] đang chạy...")

    try:
        result = rag_agent(state['question'])
        return {"rag_result": result}
    except Exception as e:
        return {"rag_result": f"[Lỗi] Không có thông tin trong tài liệu: {e}"}

In [20]:
# 3 (tiếp). Thêm try/except vào analysis_node
def analysis_node(state: OrchestratorState):
    parts = []
    
    if state["web_result"]:
        parts.append(state["web_result"])
    
    if state["financial_result"]:
        parts.append(str(state["financial_result"]))

    if state["rag_result"]:
        parts.append(state["rag_result"])

    combined_data = "\n\n".join(parts)
    try:
        analysis_result = analysis_agent(combined_data)
        print("\nAnalysis Agent: ")
        return {"final_result": analysis_result}
    except Exception as e:
        return {"final_result": f"[Lỗi] Không tổng hợp được thông tin: {e}"}

In [21]:
# 4. report_agent(question, data)
def report_agent(question, data):

    prompt = f"""Dựa vào thông tin sau đã được tổng hợp:
    {data}
    Hãy trả lời câu hỏi: {question} bằng cách xuất 1 báo cáo theo đúng 4 phần cố định như sau:
    - Tóm tắt: 1-2 câu tổng quan
    - Dữ liệu: Các số liệu chính đã thu thập được
    - Phân tích: nhận định/ xu hướng
    - Khuyến nghị: gợi ý hành động (mua/bán/theo dõi thêm...).
    Chỉ được dùng dữ liệu được cung cấp."""

    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=prompt
    )
    return response.text

In [22]:
# 5. report_node(state)
def report_node(state: OrchestratorState):
    print("[Report_Node] đang chạy...")
    try:
        result = report_agent(state['question'], state['final_result'])
        return {"report": result}
    except Exception as e:
        return {"report": f"[Lỗi] Không tạo được báo cáo: {e}"}

In [23]:
# 6. Build lại graph: thêm report_node, nối analysis -> report -> END
graph_builder = StateGraph(OrchestratorState)

graph_builder.add_node("web_research", web_research_node)
graph_builder.add_node("financial_data", financial_data_node)
graph_builder.add_node("rag", rag_node)
graph_builder.add_node("router", router_node)
graph_builder.add_node("analysis", analysis_node)
graph_builder.add_node("report", report_node)


graph_builder.add_edge(START, "router")
graph_builder.add_conditional_edges("router", route_to_agents)

graph_builder.add_edge("web_research", "analysis")
graph_builder.add_edge("financial_data", "analysis")
graph_builder.add_edge("rag", "analysis")
graph_builder.add_edge("analysis", "report")
graph_builder.add_edge("report", END)

graph = graph_builder.compile()

In [24]:
# 7. Test với nhiều câu hỏi khác nhau (trường hợp bình thường)
result1 = graph.invoke({
    "question": "Giá NVDA hiện tại bao nhiêu?",
    "symbol": "NVDA",
    "needed_agents": [],
    "web_result": "",
    "financial_result": {},
    "rag_result": "",
    "final_result": "",
    "report" : ""
})
print(result1["report"])

[Router] Agent cần chạy: ['financial_data']
[Financial Data Node] đang chạy...

Analysis Agent: 
[Report_Node] đang chạy...
Dưới đây là báo cáo về giá cổ phiếu NVDA dựa trên thông tin được cung cấp:

**BÁO CÁO CỔ PHIẾU NVIDIA (NVDA)**

**- Tóm tắt:**
Giá cổ phiếu NVIDIA (NVDA) hiện tại là 196.42. NVIDIA là một công ty công nghệ có quy mô khổng lồ, với vốn hóa thị trường hàng nghìn tỷ đô la, và thị trường đang đặt kỳ vọng cao vào tiềm năng tăng trưởng trong tương lai của công ty.

**- Dữ liệu:**
*   **Mã cổ phiếu:** NVDA (NVIDIA)
*   **Giá hiện tại (currentPrice):** 196.42
*   **Vốn hóa thị trường (marketCap):** 4,757,488,926,720 (khoảng 4.76 nghìn tỷ USD)
*   **Chỉ số P/E trailing (trailingPE):** 30.079632

**- Phân tích:**
Dựa trên dữ liệu hiện tại, giá cổ phiếu NVDA là 196.42. Với vốn hóa thị trường cực lớn xấp xỉ 4.76 nghìn tỷ USD, NVIDIA đã khẳng định vị thế là một trong những công ty có giá trị nhất thế giới. Chỉ số P/E trailing ở mức 30.08 được coi là tương đối cao, cho thấy thị 

In [25]:
# Câu hỏi 2: fan-out 2 agent (web_research + rag)
result2 = graph.invoke({
    "question": "Doanh thu NVIDIA quý 1 2027 là bao nhiêu, có tin tức gì mới không?",
    "symbol": "NVDA",
    "needed_agents": [],
    "web_result": "",
    "financial_result": {},
    "rag_result": "",
    "final_result": "",
    "report": ""
})
print(result2["report"])

[Router] Agent cần chạy: ['web_research', 'rag']
[Rag Node] đang chạy...
[Web Research Node] đang chạy...

Analysis Agent: 
[Report_Node] đang chạy...
Dưới đây là báo cáo về doanh thu và tin tức mới của NVIDIA:

---

### Báo cáo về NVIDIA: Quý 1 năm 2027

**- Tóm tắt:**
NVIDIA đã báo cáo doanh thu kỷ lục 81.6 tỷ USD cho quý 1 năm 2027, cho thấy hiệu suất tài chính vượt trội được thúc đẩy chủ yếu bởi mảng Data Center và những đổi mới liên tục trong hệ sinh thái AI toàn diện.

**- Dữ liệu:**
*   **Doanh thu quý 1 năm 2027:** 81.6 tỷ USD.
*   **Tăng trưởng doanh thu:** Tăng 20% so với quý trước và 85% so với cùng kỳ năm ngoái.
*   **Doanh thu mảng Data Center:** Đạt kỷ lục 75.2 tỷ USD, tăng 92% so với năm trước, là động lực tăng trưởng chính.
*   **Chính sách hoàn trả cổ đông:** Ủy quyền mua lại cổ phiếu bổ sung 80.0 tỷ USD và tăng cổ tức tiền mặt hàng quý từ 0.01 USD lên 0.25 USD/cổ phiếu.
*   **Đổi mới sản phẩm nổi bật:** Phát hành DLSS 4.5 Dynamic Multi Frame Generation, giới thiệu DLS

In [26]:
# Câu hỏi 3: không khớp keyword nào -> mặc định chạy cả 3 agent
result3 = graph.invoke({
    "question": "Bạn nhận định chung thế nào về NVIDIA?",
    "symbol": "NVDA",
    "needed_agents": [],
    "web_result": "",
    "financial_result": {},
    "rag_result": "",
    "final_result": "",
    "report": ""
})
print(result3["report"])

[Router] Agent cần chạy: ['web_research', 'financial_data', 'rag']
[Financial Data Node] đang chạy...
[Rag Node] đang chạy...
[Web Research Node] đang chạy...

Analysis Agent: 
[Report_Node] đang chạy...
Dựa trên thông tin được cung cấp, dưới đây là báo cáo về NVIDIA:

**- Tóm tắt:**
NVIDIA đang thể hiện một xu hướng tăng trưởng tài chính bùng nổ và bền vững, củng cố vị thế dẫn đầu không thể tranh cãi trong cuộc cách mạng AI toàn cầu. Công ty liên tục tăng tốc doanh thu, lợi nhuận và mở rộng vị thế chiến lược, đồng thời cam kết mạnh mẽ tạo ra giá trị cho cổ đông.

**- Dữ liệu:**
*   Doanh thu Q1 FY27: $81.6 tỷ, tăng 85% so với cùng kỳ năm trước (YoY) và 20% so với quý trước (QoQ).
*   Doanh thu từ Trung tâm Dữ liệu: $75.2 tỷ, tăng 92% YoY.
*   Doanh thu mạng lưới trung tâm dữ liệu: tăng 199% YoY.
*   Tỷ suất lợi nhuận gộp (GAAP): 74.9%, tăng 14.4 điểm phần trăm YoY.
*   Lợi nhuận ròng: tăng hơn 200% YoY.
*   Tài sản: tăng từ $206.803 triệu lên $259.474 triệu chỉ trong một quý.
*   Ủy q

In [28]:
# 7 (tiếp). Test error handling: gây lỗi có chủ đích, xác nhận graph không sập

# Lưu lại hàm gốc để khôi phục sau khi test
_original_financial_data_agent = financial_data_agent


def _broken_financial_data_agent(symbol):
    raise ValueError("Giả lập lỗi gọi API (test error handling)")


# Gán đè hàm bị lỗi vào tên financial_data_agent
# financial_data_node bên trong vẫn gọi đúng tên "financial_data_agent" -> sẽ gọi hàm lỗi này
financial_data_agent = _broken_financial_data_agent

result_error_test = graph.invoke({
    "question": "Giá NVDA hiện tại bao nhiêu?",
    "symbol": "NVDA",
    "needed_agents": [],
    "web_result": "",
    "financial_result": {},
    "rag_result": "",
    "final_result": "",
    "report": ""
})
print(result_error_test["report"])

# Khôi phục lại hàm gốc, không ảnh hưởng các lần test khác
financial_data_agent = _original_financial_data_agent

[Router] Agent cần chạy: ['financial_data']
[Financial Data Node] đang chạy...

Analysis Agent: 
[Report_Node] đang chạy...
Dựa trên thông tin đã được cung cấp, dưới đây là báo cáo theo yêu cầu:

**Báo cáo về Giá cổ phiếu NVDA hiện tại**

**- Tóm tắt:**
Không thể xác định giá NVDA hiện tại do không có dữ liệu tài chính nào được cung cấp. Thông tin duy nhất có sẵn là một thông báo lỗi về việc không truy xuất được dữ liệu.

**- Dữ liệu:**
Dữ liệu duy nhất thu thập được là thông báo lỗi: "[Lỗi] Không lấy có dữ liệu: Giả lập lỗi gọi API (test error handling)". Không có bất kỳ số liệu tài chính cụ thể nào về giá cổ phiếu, hiệu suất hay các chỉ số khác của NVIDIA được cung cấp.

**- Phân tích:**
Do chỉ nhận được thông báo lỗi, không có số liệu hay xu hướng tài chính nào của NVDA có thể được phân tích. Việc thiếu dữ dữ liệu ngăn cản mọi nhận định về hiệu suất hay biến động giá cổ phiếu. Không thể đưa ra bất kỳ kết luận nào về tình hình tài chính hoặc giá trị hiện tại của cổ phiếu NVDA.

**- K